In [15]:
import pandas as pd

data = pd.read_csv("original_blocks/original_block_363.csv")
def check_duplicates(data: pd.DataFrame):
        try:
            print("Grouping genes with identical expression values...")
            # Transpose to check duplicates across genes
            genes_duplicated = data.T[data.T.duplicated(keep=False)].T
            if genes_duplicated.empty:
                print("No duplicate genes found")
                return {}, {}

            # Map each gene to its expression values
            gene_values = {col: vals.tolist() for col, vals in genes_duplicated.items()}
            duplicated_genes_dict = {col: [] for col in genes_duplicated.columns}
            mapped_genes_dict = duplicated_genes_dict.copy()

            # Group duplicates by expression values
            seen_values = set()
            for column in tqdm(genes_duplicated.columns, desc="Processing duplicates"):
                values = tuple(gene_values[column])  # Use tuple for hashability
                if values in seen_values:
                    continue
                seen_values.add(values)
                duplicates = [col for col, vals in gene_values.items() if vals == gene_values[column]]
                for dup in duplicates:
                    duplicated_genes_dict[dup] = duplicates

            # Save results
            with open("duplicated_genes_dict.json", "w") as f:
                json.dump(duplicated_genes_dict, f) 
            return duplicated_genes_dict
        except Exception as e:
            raise ValueError(f"Duplicate check failed: {e}")

In [17]:

dup_mask_numeric = data.T.duplicated(keep='first')
keep_mask_all = pd.Series(True, index=data.columns)
keep_mask_all.loc[dup_mask_numeric[dup_mask_numeric].index] = False
# DataFrame sau khi loại duplicate chỉ trong num_cols
df_cleaned_identical = data.loc[:, keep_mask_all]

print(df_cleaned_identical.shape) 

# Get removed features (all columns)
removed_features = keep_mask_all[~keep_mask_all].index.tolist()
identical_df = data[removed_features]
# identical_df.to_csv("avatars/identical_genes.csv", index = True)

(311, 3701)


In [14]:
identical_df["ENSG00000207511.1"].var()

0.534451559149987